In [1]:
import numpy as np
import geopandas as gpd
import pandas as pd
from tqdm import tqdm
from shapely.geometry import Polygon, Point
from shapely.vectorized import contains  # use contains_xy se estiver com Shapely >= 2.0

def define_geo_points(polygon,num_required):
    minx, miny, maxx, maxy = polygon.bounds
    area_bbox = (maxx - minx) * (maxy - miny)
    area_polygon = polygon.area

    # Gerar pontos vetorialmente
    batch_size = int(num_required /(area_polygon/area_bbox))  # margem para garantir cobertura
    a = 2.326  # z para 99%
    batch_size = int((num_required + a /2 + a*(num_required + a/4)**0.5)/(area_polygon/area_bbox))  # margem para garantir cobertura
    x = np.random.uniform(minx, maxx, batch_size)
    y = np.random.uniform(miny, maxy, batch_size)
    
    # Filtrar vetorialmente os pontos dentro do polígono
    mask = contains(polygon, x, y)  # use contains_xy se estiver com Shapely >= 2.0
    valid_coords = np.column_stack((x[mask], y[mask]))

    # Selecionar os primeiros 55.000 pontos válidos
    selected_coords = valid_coords[:int(num_required)]
    
    if len(valid_coords) < num_required:
        print(batch_size, len(valid_coords))
        selected_coords = define_geo_points(polygon,num_required)
    
    return selected_coords



In [2]:
AGR_DOM = pd.read_csv('1600501_OIAPOQUE/Agregados_por_setores_caracteristicas_domicilio1_BR.csv', sep = ';')
cods = [str(a) for a in AGR_DOM.CD_setor]
AGR_DOM = AGR_DOM.replace('X', '0').astype(int)
AGR_DOM['COD_setor'] = cods#[str(a) for a in AGR_DOM.CD_setor]
AGR_DOM = AGR_DOM.drop(columns=['CD_setor'])
AGR_DOM

C:\Users\carlo\AppData\Local\Temp\ipykernel_30708\3392966801.py:1: DtypeWarning: Columns (1,3,4,5,7,8,9,10,11,13,14,15,16,17,20,21,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,51,52,53,54,56,57,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,77,78,84,87) have mixed types. Specify dtype option on import or set low_memory=False.
  AGR_DOM = pd.read_csv('1600501_OIAPOQUE/Agregados_por_setores_caracteristicas_domicilio1_BR.csv', sep = ';')


,V00001,V00002,V00003,V00004,V00005,V00006,V00007,V00008,V00009,V00010,...,V00081,V00082,V00083,V00084,V00085,V00086,V00087,V00088,V00089,COD_setor
0,336,0,0,336,928,0,0,136,0,0,...,0,0,0,926,0,0,136,0,0,110001505000002
1,208,0,0,208,556,0,0,73,0,0,...,0,0,0,556,0,0,73,0,0,110001505000003
2,85,0,0,85,222,0,0,24,0,0,...,0,0,0,222,0,0,24,0,0,110001505000004
3,281,0,0,281,783,0,0,124,0,0,...,0,0,0,780,0,3,123,0,0,110001505000006
4,291,0,0,291,748,0,0,102,0,0,...,0,0,0,748,0,0,102,0,0,110001505000007
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
458767,10,0,0,10,36,0,0,7,0,0,...,0,0,0,36,0,0,7,0,0,530010805440139
458768,227,0,0,227,632,0,0,79,0,0,...,0,0,0,620,4,8,75,0,0,530010805440140
458769,145,0,0,145,387,0,0,52,0,0,...,0,0,0,313,0,73,38,0,14,530010805440141
458770,107,0,0,107,348,0,0,32,0,0,...,0,0,0,77,267,4,10,20,0,530010805440142


In [3]:
setores = gpd.read_file('1600501_OIAPOQUE/AP_setores_CD2022.shp')
setores = setores[setores['NM_MUN']=='Oiapoque']
setores

,CD_SETOR,SITUACAO,CD_SIT,CD_TIPO,AREA_KM2,CD_REGIAO,NM_REGIAO,CD_UF,NM_UF,CD_MUN,...,NM_FCU,CD_AGLOM,NM_AGLOM,CD_RGINT,NM_RGINT,CD_RGI,NM_RGI,CD_CONCURB,NM_CONCURB,geometry
1083,160050105000001,Urbana,1,0,0.117806,1,Norte,16,Amapá,1600501,...,None,None,None,1602,Oiapoque - Porto Grande,160003,Oiapoque,None,None,"POLYGON ((-51.83259 3.84679, -51.83315 3.84813..."
1084,160050105000004,Urbana,1,0,0.215701,1,Norte,16,Amapá,1600501,...,None,None,None,1602,Oiapoque - Porto Grande,160003,Oiapoque,None,None,"POLYGON ((-51.83491 3.84363, -51.83539 3.84338..."
1085,160050105000007,Rural,5,5,0.706869,1,Norte,16,Amapá,1600501,...,None,160050100004,Aldeia Indígena São José dos Galibis,1602,Oiapoque - Porto Grande,160003,Oiapoque,None,None,"POLYGON ((-51.76159 3.97477, -51.76783 3.96367..."
1086,160050105000008,Rural,8,0,69.623041,1,Norte,16,Amapá,1600501,...,None,None,None,1602,Oiapoque - Porto Grande,160003,Oiapoque,None,None,"POLYGON ((-51.69066 3.96071, -51.69301 3.95662..."
1087,160050105000009,Rural,8,0,603.031833,1,Norte,16,Amapá,1600501,...,None,None,None,1602,Oiapoque - Porto Grande,160003,Oiapoque,None,None,"POLYGON ((-51.63154 4.0493, -51.63175 4.04934,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1175,160050115000029,Rural,8,5,0.395323,1,Norte,16,Amapá,1600501,...,None,160050100054,Aldeia Indígena Kaxiuahí,1602,Oiapoque - Porto Grande,160003,Oiapoque,None,None,"POLYGON ((-51.34329 3.30718, -51.34315 3.30778..."
1176,160050115000030,Rural,8,0,1588.647167,1,Norte,16,Amapá,1600501,...,None,None,None,1602,Oiapoque - Porto Grande,160003,Oiapoque,None,None,"POLYGON ((-51.53538 3.33891, -51.52628 3.34307..."
1177,160050120000001,Rural,5,0,1.169275,1,Norte,16,Amapá,1600501,...,None,160050100014,Vila Brasil,1602,Oiapoque - Porto Grande,160003,Oiapoque,None,None,"POLYGON ((-52.3231 3.17078, -52.32306 3.15922,..."
1178,160050120000002,Rural,8,0,79.760938,1,Norte,16,Amapá,1600501,...,None,None,None,1602,Oiapoque - Porto Grande,160003,Oiapoque,None,None,"POLYGON ((-52.23693 3.23941, -52.23329 3.23565..."


In [4]:
CNEFE = pd.read_csv('1600501_OIAPOQUE/1600501_OIAPOQUE.csv', sep = ';')
CNEFE['COD_setor'] = [a[:-1] for a in CNEFE.COD_SETOR]
CNEFE['ponto'] = gpd.points_from_xy(CNEFE['LONGITUDE'], CNEFE['LATITUDE'], crs='EPSG:4326')
CNEFE

,COD_UNICO_ENDERECO,COD_UF,COD_MUNICIPIO,COD_DISTRITO,COD_SUBDISTRITO,COD_SETOR,NUM_QUADRA,NUM_FACE,CEP,DSC_LOCALIDADE,...,LONGITUDE,NV_GEO_COORD,COD_ESPECIE,DSC_ESTABELECIMENTO,COD_INDICADOR_ESTAB_ENDERECO,COD_INDICADOR_CONST_ENDERECO,COD_INDICADOR_FINALIDADE_CONST,COD_TIPO_ESPECI,COD_setor,ponto
0,7002596,16,1600501,160050105,16005010500,160050105000021P,14,4,68980000,CENTRO,...,-51.830668,1,1,NaN,NaN,NaN,NaN,101.0,160050105000021,POINT (-51.83067 3.84618)
1,7002598,16,1600501,160050105,16005010500,160050105000021P,14,4,68980000,CENTRO,...,-51.830645,1,7,NaN,NaN,1.0,4.0,NaN,160050105000021,POINT (-51.83064 3.84599)
2,7002599,16,1600501,160050105,16005010500,160050105000021P,14,4,68980000,CENTRO,...,-51.830685,1,1,NaN,NaN,NaN,NaN,101.0,160050105000021,POINT (-51.83068 3.84635)
3,7002600,16,1600501,160050105,16005010500,160050105000021P,14,4,68980000,CENTRO,...,-51.830810,1,1,NaN,NaN,NaN,NaN,101.0,160050105000021,POINT (-51.83081 3.8465)
4,7002601,16,1600501,160050105,16005010500,160050105000021P,14,4,68980000,CENTRO,...,-51.830846,1,6,BARBEARIA SARGES,1.0,NaN,NaN,NaN,160050105000021,POINT (-51.83085 3.84661)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12623,215441896,16,1600501,160050105,16005010500,160050105000001P,6,2,68980000,CENTRO,...,-51.831284,1,1,NaN,NaN,NaN,NaN,103.0,160050105000001,POINT (-51.83128 3.84761)
12624,215441897,16,1600501,160050105,16005010500,160050105000001P,6,2,68980000,CENTRO,...,-51.831300,1,1,NaN,NaN,NaN,NaN,103.0,160050105000001,POINT (-51.8313 3.8476)
12625,215441898,16,1600501,160050105,16005010500,160050105000001P,6,2,68980000,CENTRO,...,-51.831490,1,1,NaN,NaN,NaN,NaN,103.0,160050105000001,POINT (-51.83149 3.84737)
12626,215441899,16,1600501,160050105,16005010500,160050105000001P,6,2,68980000,CENTRO,...,-51.831276,1,1,NaN,NaN,NaN,NaN,101.0,160050105000001,POINT (-51.83128 3.84761)


In [5]:
simulados = []

# Pernamentes

agr_per = AGR_DOM[['COD_setor','V00017', 'V00018', 'V00019', 'V00020', 'V00021', 'V00022', 'V00023', 'V00024', 'V00025', 'V00026',
         'V00047','V00048','V00049','V00001']].rename(columns={'V00017':'m1', 'V00018':'m2', 'V00019':'m3', 'V00020':'m4', 
                                                                                 'V00021':'m5', 'V00022':'m6', 'V00023':'m7', 'V00024':'m8', 'V00025':'m9', 'V00026':'m10',
                                                                                 'V00047':'casa','V00048':'vila','V00049':'ap','V00001':'total'})
agr_per['diff'] = agr_per.total-agr_per.casa-agr_per.vila-agr_per.ap
agr_per['sem_num_moradores'] = agr_per.total-agr_per.m1-agr_per.m2-agr_per.m3-agr_per.m4-agr_per.m5-agr_per.m6-agr_per.m7-agr_per.m8-agr_per.m9-agr_per.m10
agr_per

lista_domicilios_permanentes_df = []

s = setores.merge(agr_per, left_on='CD_SETOR', right_on='COD_setor', how='inner')

for idx_setores in tqdm(range(len(s))):
    domicilios_permanentes = []
    for especie in [('casa',101), ('vila',102), ('ap',103)]:
        cod_setor = s.loc[idx_setores].CD_SETOR
        num_domicilios = s.loc[idx_setores][especie[0]]
        geo = s.loc[idx_setores].geometry
        
        if num_domicilios == 0:
            continue

        dic_default = {'cod_setor': cod_setor, 'tipo': 'permanente', 'especie': especie[0]}

        CNEFE_setor = CNEFE[(CNEFE.COD_setor == cod_setor) & (CNEFE.COD_ESPECIE == 1) & (CNEFE.COD_TIPO_ESPECI == especie[1])]
        if len(CNEFE_setor) > 0:
            CNEFE_setor['within'] = CNEFE_setor.apply(
                lambda row: row['ponto'].within(geo),
                axis=1
            )
            CNEFE_setor = CNEFE_setor[CNEFE_setor.within == True]

        lista_CNEFE = [dic_default | {'fonte': 'CNEFE', 'endereco': x} for x in CNEFE_setor.COD_UNICO_ENDERECO.tolist()]
        if len(lista_CNEFE) >= num_domicilios:
            indices_selecionados = np.random.choice(len(lista_CNEFE), size=num_domicilios, replace=False)
            domicilios_permanentes = domicilios_permanentes + [lista_CNEFE[i] for i in indices_selecionados]
        else:
            lista_faltante = define_geo_points(geo, num_domicilios - len(lista_CNEFE))
            simulados = simulados + [dic_default | {'id': x, 'ponto':lista_faltante[x]} for x in range(len(lista_faltante))]
            lista_final = lista_CNEFE + [dic_default | {'fonte': 'simulados', 'endereco': x} for x in range(len(lista_faltante))]
            indices_selecionados = np.random.choice(len(lista_final), size=num_domicilios, replace=False)
            domicilios_permanentes = domicilios_permanentes + [lista_final[i] for i in indices_selecionados]

    domicilios_permanentes_df = pd.DataFrame(domicilios_permanentes)

    num_diff = s.loc[idx_setores]['diff']
    if num_diff > 0:

        dic_default = {'cod_setor': cod_setor, 'tipo': 'permanente', 'especie': 'outros'}

        CNEFE_setor = CNEFE[
            (~CNEFE.COD_UNICO_ENDERECO.isin(domicilios_permanentes_df[domicilios_permanentes_df['fonte'] == 'CNEFE'].endereco.to_list())) & 
            (CNEFE.COD_ESPECIE == 1) & 
            (CNEFE.COD_setor == cod_setor)]
        
        if len(CNEFE_setor) > 0:
            CNEFE_setor['within'] = CNEFE_setor.apply(
                    lambda row: row['ponto'].within(geo),
                    axis=1
                )
            CNEFE_setor = CNEFE_setor[CNEFE_setor.within == True]
        lista_CNEFE = [dic_default | {'fonte': 'CNEFE', 'endereco': x} for x in CNEFE_setor.COD_UNICO_ENDERECO.tolist()]
        if len(lista_CNEFE) >= num_diff:
            indices_selecionados = np.random.choice(len(lista_CNEFE), size=num_diff, replace=False)
            novos = [lista_CNEFE[i] for i in indices_selecionados]
        else:
            lista_faltante = define_geo_points(geo, num_diff - len(lista_CNEFE))
            simulados = simulados + [dic_default | {'id': x, 'ponto':lista_faltante[x]} for x in range(len(lista_faltante))]
            lista_final = lista_CNEFE + [dic_default | {'fonte': 'simulados', 'endereco': x} for x in range(len(lista_faltante))]
            indices_selecionados = np.random.choice(len(lista_final), size=num_diff, replace=False)
            novos = [lista_final[i] for i in indices_selecionados]

        domicilios_permanentes_df = pd.DataFrame(domicilios_permanentes + novos)

    # quantidade de moradores por domicilio
    lista_num_moradores = []
    for num_moradores in ['m1', 'm2', 'm3', 'm4', 'm5', 'm6', 'm7', 'm8', 'm9', 'm10', 'sem_num_moradores']:
        lista_num_moradores += [num_moradores]*s.loc[idx_setores][num_moradores]
    

    domicilios_permanentes_df['num_moradores'] = np.random.choice(lista_num_moradores, size=len(domicilios_permanentes_df), replace=False)

    lista_domicilios_permanentes_df.append(domicilios_permanentes_df)


# Improvisados

agr_imp = AGR_DOM[['COD_setor','V00027', 'V00028', 'V00029', 'V00030', 'V00031', 'V00032', 'V00033', 'V00034', 'V00035', 'V00036',
'V00053','V00054','V00055','V00056','V00057','V00058','V00002']].rename(columns={'V00027':'m1', 'V00028':'m2', 'V00029':'m3', 'V00030':'m4', 
                                                                'V00031':'m5', 'V00032':'m6', 'V00033':'m7', 'V00034':'m8', 'V00035':'m9', 'V00036':'m10',
                                                                'V00053':'tenda','V00054':'estabelecimento','V00055':'natural','V00056':'publico',
                                                                'V00057':'n_residencial', 'V00058':'veiculo', 'V00002':'total'})
agr_imp['diff'] = agr_imp.total-agr_imp.tenda-agr_imp.estabelecimento-agr_imp.natural-agr_imp.publico-agr_imp.n_residencial-agr_imp.veiculo
agr_imp['sem_num_moradores'] = agr_imp.total-agr_imp.m1-agr_imp.m2-agr_imp.m3-agr_imp.m4-agr_imp.m5-agr_imp.m6-agr_imp.m7-agr_imp.m8-agr_imp.m9-agr_imp.m10

lista_domicilios_improvisados_df = []

s = setores.merge(agr_imp, left_on='CD_SETOR', right_on='COD_setor', how='inner')

for idx_setores in tqdm(range(len(s))):

    cod_setor = s.loc[idx_setores].CD_SETOR
    num_domicilios = s.loc[idx_setores]['total']
    geo = s.loc[idx_setores].geometry

    dic_default = {'cod_setor': cod_setor, 'tipo': 'improvisado', 'especie': ''}

    lista_especies = []
    for especie in ('tenda', 'estabelecimento', 'natural', 'publico', 'n_residencial', 'veiculo','diff'):
        especie_name = 'outros' if especie == 'diff' else especie
        lista_especies += [especie_name]*s.loc[idx_setores][especie]

    lista_faltante = define_geo_points(geo, num_domicilios)
    simulados = simulados + [dic_default | {'especie':lista_especies[x], 'id': x, 'ponto':lista_faltante[x]} for x in range(len(lista_faltante))]
    lista_selecionados = [dic_default | {'especie':lista_especies[x], 'fonte': 'simulados', 'endereco': x} for x in range(len(lista_faltante))]

    domicilios_improvisados_df = pd.DataFrame(lista_selecionados)
    lista_num_moradores = []
    for num_moradores in ['m1', 'm2', 'm3', 'm4', 'm5', 'm6', 'm7', 'm8', 'm9', 'm10', 'sem_num_moradores']:
        lista_num_moradores += [num_moradores]*s.loc[idx_setores][num_moradores]
    domicilios_improvisados_df['num_moradores'] = np.random.choice(lista_num_moradores, size=len(domicilios_improvisados_df), replace=False)

    lista_domicilios_improvisados_df.append(domicilios_improvisados_df)


# Coletivos

agr_col = AGR_DOM[['COD_setor','V00037', 'V00038', 'V00039', 'V00040', 'V00041', 'V00042', 'V00043', 'V00044', 'V00045', 'V00046',
'V00059','V00060','V00061','V00062','V00063','V00064','V00065','V00066','V00067','V00068','V00069','V00003']].rename(
    columns={'V00037':'m1', 'V00038':'m2', 'V00039':'m3', 'V00040':'m4', 'V00041':'m5', 'V00042':'m6', 'V00043':'m7', 'V00044':'m8', 'V00045':'m9', 'V00046':'m10',
            'V00059':'asilo','V00060':'hotel','V00061':'alojamento','V00062':'penitenciaria', 'V00063':'outros_domicilios', 'V00064':'albergue', 
            'V00065':'abrigo','V00066':'clinica_psi','V00067':'orfanato','V00068':'internacao_menores','V00069':'quartel','V00003':'total'})

agr_col['diff'] = agr_col.total-agr_col.asilo-agr_col.hotel-agr_col.alojamento-agr_col.penitenciaria-agr_col.outros_domicilios-agr_col.albergue-agr_col.abrigo-agr_col.clinica_psi-agr_col.orfanato-agr_col.internacao_menores-agr_col.quartel
agr_col['sem_num_moradores'] = agr_col.total-agr_col.m1-agr_col.m2-agr_col.m3-agr_col.m4-agr_col.m5-agr_col.m6-agr_col.m7-agr_col.m8-agr_col.m9-agr_col.m10

lista_domicilios_coletivos_df = []

s = setores.merge(agr_col, left_on='CD_SETOR', right_on='COD_setor', how='inner')

for idx_setores in tqdm(range(len(s))):

    cod_setor = s.loc[idx_setores].CD_SETOR
    num_domicilios = s.loc[idx_setores]['total']
    geo = s.loc[idx_setores].geometry

    dic_default = {'cod_setor': cod_setor, 'tipo': 'coletivo', 'especie': ''}

    CNEFE_setor = CNEFE[(CNEFE.COD_setor == cod_setor) & (CNEFE.COD_ESPECIE == 2)]
    if len(CNEFE_setor) > 0:
        CNEFE_setor['within'] = CNEFE_setor.apply(
                lambda row: row['ponto'].within(geo),
                axis=1
            )
        CNEFE_setor = CNEFE_setor[CNEFE_setor.within == True]

    lista_especies = []
    for especie in ('asilo', 'hotel', 'alojamento', 'penitenciaria', 'outros_domicilios', 'albergue', 'abrigo', 'clinica_psi', 'orfanato', 'internacao_menores', 'quartel', 'diff'):
        especie_name = 'outros' if especie == 'diff' else especie
        lista_especies += [especie_name]*s.loc[idx_setores][especie]
    lista_especies = np.random.choice(lista_especies, size=num_domicilios, replace=False)

    lista_CNEFE = [dic_default | {'fonte': 'CNEFE', 'endereco': x} for x in CNEFE_setor.COD_UNICO_ENDERECO.tolist()]
    if len(lista_CNEFE) >= num_domicilios:
        indices_selecionados = np.random.choice(len(lista_CNEFE), size=num_domicilios, replace=False)
        lista_selecionados = [lista_CNEFE[i] for i in indices_selecionados]
    else:
        lista_faltante = define_geo_points(geo, num_domicilios - len(lista_CNEFE))
        simulados = simulados + [dic_default | {'especie': lista_especies[x], 'id': x, 'ponto':lista_faltante[x]} for x in range(len(lista_faltante))]
        # aqui, a ordem importa, pois para o especie é chave primariado simulados
        lista_selecionados =[dic_default | {'fonte': 'simulados', 'endereco': x} for x in range(len(lista_faltante))] + lista_CNEFE
 

    domicilios_coletivos_df = pd.DataFrame(lista_selecionados)

    domicilios_coletivos_df['especie'] = lista_especies
    lista_num_moradores = []
    for num_moradores in ['m1', 'm2', 'm3', 'm4', 'm5', 'm6', 'm7', 'm8', 'm9', 'm10', 'sem_num_moradores']:
        lista_num_moradores += [num_moradores]*s.loc[idx_setores][num_moradores]
    domicilios_coletivos_df['num_moradores'] = np.random.choice(lista_num_moradores, size=len(domicilios_coletivos_df), replace=False)

    lista_domicilios_coletivos_df.append(domicilios_coletivos_df)


  0%|          | 0/92 [00:00<?, ?it/s]C:\Users\carlo\AppData\Local\Temp\ipykernel_30708\3263687940.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  CNEFE_setor['within'] = CNEFE_setor.apply(
C:\Users\carlo\AppData\Local\Temp\ipykernel_30708\3263687940.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  CNEFE_setor['within'] = CNEFE_setor.apply(
C:\Users\carlo\AppData\Local\Temp\ipykernel_30708\3263687940.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.


In [6]:
simulados_df = pd.DataFrame(simulados)
simulados_df

,cod_setor,tipo,especie,id,ponto
0,160050105000008,permanente,casa,0,"[-51.76198423482908, 3.979716844487329]"
1,160050105000008,permanente,casa,1,"[-51.73051830314784, 3.9950191223046057]"
2,160050105000061,permanente,casa,0,"[-51.80534051722232, 3.8390067571270605]"
3,160050105000061,permanente,casa,1,"[-51.80577066274322, 3.839486886966274]"
4,160050105000061,permanente,casa,2,"[-51.80437309487463, 3.838838006477474]"
...,...,...,...,...,...
4439,160050120000003,improvisado,outros,7,"[-52.25706005527198, 3.245395085958301]"
4440,160050105000025,coletivo,hotel,0,"[-51.83149574873098, 3.838322509772568]"
4441,160050105000110,coletivo,hotel,0,"[-51.83461566846084, 3.8482883469489466]"
4442,160050105000110,coletivo,hotel,1,"[-51.8344881348391, 3.8408742044192166]"


In [63]:
domicilios = pd.concat(lista_domicilios_permanentes_df + lista_domicilios_improvisados_df + lista_domicilios_coletivos_df, ignore_index=True)
domicilios

,cod_setor,tipo,especie,fonte,endereco,num_moradores
0,160050105000001,permanente,casa,CNEFE,6998898.0,m5
1,160050105000001,permanente,casa,CNEFE,6998926.0,m2
2,160050105000001,permanente,casa,CNEFE,6998878.0,m2
3,160050105000001,permanente,casa,CNEFE,215441932.0,m4
4,160050105000001,permanente,casa,CNEFE,215441963.0,m5
...,...,...,...,...,...,...
7107,160050120000001,coletivo,outros,CNEFE,182416921.0,m1
7108,160050120000001,coletivo,hotel,CNEFE,182417066.0,sem_num_moradores
7109,160050120000003,coletivo,alojamento,CNEFE,182400623.0,sem_num_moradores
7110,160050120000003,coletivo,alojamento,CNEFE,7005316.0,sem_num_moradores


In [64]:
domicilios['num_moradores_int'] = domicilios['num_moradores'].replace({'sem_num_moradores': 'm1'}).apply(lambda x: np.nan if pd.isna(x) else int(x[1:]) )
domicilios['mais_moradores'] = domicilios['num_moradores'].isin(['sem_num_moradores','m10'])
domicilios = domicilios.reset_index().rename(columns={'index':'id_domicilio'})
domicilios

,id_domicilio,cod_setor,tipo,especie,fonte,endereco,num_moradores,num_moradores_int,mais_moradores
0,0,160050105000001,permanente,casa,CNEFE,6998898.0,m5,5,False
1,1,160050105000001,permanente,casa,CNEFE,6998926.0,m2,2,False
2,2,160050105000001,permanente,casa,CNEFE,6998878.0,m2,2,False
3,3,160050105000001,permanente,casa,CNEFE,215441932.0,m4,4,False
4,4,160050105000001,permanente,casa,CNEFE,215441963.0,m5,5,False
...,...,...,...,...,...,...,...,...,...
7107,7107,160050120000001,coletivo,outros,CNEFE,182416921.0,m1,1,False
7108,7108,160050120000001,coletivo,hotel,CNEFE,182417066.0,sem_num_moradores,1,True
7109,7109,160050120000003,coletivo,alojamento,CNEFE,182400623.0,sem_num_moradores,1,True
7110,7110,160050120000003,coletivo,alojamento,CNEFE,7005316.0,sem_num_moradores,1,True


In [65]:
agr_dom_pes = AGR_DOM[['COD_setor','V00008','V00009','V00010','V00011','V00012','V00013', 'V00014','V00015','V00016']].rename(
    columns={'V00008':'per_crianca','V00009':'imp_crianca','V00010':'col_crianca','V00011':'per_homem','V00012':'imp_homem','V00013':'col_homem',
             'V00014':'per_mulher','V00015':'imp_mulher','V00016':'col_mulher'})

setores_pes = setores[['CD_SETOR']].merge(agr_dom_pes, left_on='CD_SETOR', right_on='COD_setor', how='inner')
setores_pes

,CD_SETOR,COD_setor,per_crianca,imp_crianca,col_crianca,per_homem,imp_homem,col_homem,per_mulher,imp_mulher,col_mulher
0,160050105000001,160050105000001,115,0,0,347,9,0,341,5,0
1,160050105000004,160050105000004,129,0,0,425,0,0,385,0,0
2,160050105000007,160050105000007,7,0,0,13,0,0,12,0,0
3,160050105000008,160050105000008,0,0,0,10,0,0,8,0,0
4,160050105000009,160050105000009,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
87,160050115000029,160050115000029,8,0,0,13,0,0,10,0,0
88,160050115000030,160050115000030,0,0,0,0,0,0,0,0,0
89,160050120000001,160050120000001,33,4,0,107,14,10,92,12,5
90,160050120000002,160050120000002,5,0,0,18,0,0,11,0,0


In [66]:
pessoas = pd.read_csv('1600501_OIAPOQUE/listagem_individuos_sinteticos.csv', sep=';')
pessoas['COD_setor'] = pessoas.COD_setor.astype(str)
pessoas

,COD_setor,sexo,faixa_etaria,tipo_domicilio,id_domicilio
0,160050105000001,M,3,NaN,NaN
1,160050105000001,M,3,NaN,NaN
2,160050105000001,F,4,NaN,NaN
3,160050105000001,F,0a,NaN,NaN
4,160050105000001,M,6,NaN,NaN
...,...,...,...,...,...
27307,160050120000003,F,3,NaN,NaN
27308,160050120000003,M,5,NaN,NaN
27309,160050120000003,M,2a,NaN,NaN
27310,160050120000003,M,7,NaN,NaN


In [67]:
pessoas = pd.read_csv('1600501_OIAPOQUE/listagem_individuos_sinteticos.csv', sep=';')
pessoas['COD_setor'] = pessoas.COD_setor.astype(str)
pessoas

for index, s in setores_pes.iterrows():
    for tipo in ['permanente', 'improvisado', 'coletivo']:
        tip = tipo[:3]
        num_crianca = s[f'{tip}_crianca']
        num_homem = s[f'{tip}_homem']
        num_mulher = s[f'{tip}_mulher']

        cod_setor = s['COD_setor']
        index_crianca = pessoas[(pessoas['COD_setor'] == cod_setor) & (pessoas['faixa_etaria'].isin(['0a','0b'])) & (pessoas['tipo_domicilio'].isna())].iloc[:num_crianca].index
        pessoas.loc[index_crianca, 'tipo_domicilio'] = tipo

        num_homem -= sum((pessoas.tipo_domicilio == tipo) & (pessoas.sexo == 'M') & (pessoas['COD_setor'] == cod_setor))
        num_mulher -= sum((pessoas.tipo_domicilio == tipo) & (pessoas.sexo == 'F') & (pessoas['COD_setor'] == cod_setor))

        index_homem = pessoas[(pessoas['COD_setor'] == cod_setor) & (pessoas.sexo == 'M') & (~pessoas['faixa_etaria'].isin(['0a','0b'])) & (pessoas['tipo_domicilio'].isna())].iloc[:num_homem].index
        pessoas.loc[index_homem, 'tipo_domicilio'] = tipo

        index_mulher = pessoas[(pessoas['COD_setor'] == cod_setor) & (pessoas.sexo == 'F') & (~pessoas['faixa_etaria'].isin(['0a','0b'])) & (pessoas['tipo_domicilio'].isna())].iloc[:num_mulher].index
        pessoas.loc[index_mulher, 'tipo_domicilio'] = tipo

        domicilios_setor = domicilios[(domicilios.cod_setor == cod_setor) & (domicilios.tipo == tipo)]

        #primeiro integrante de cada domicilio
        id_domicilio_list = domicilios_setor.id_domicilio.tolist()
        index_primeiro_morador = pessoas[(pessoas['COD_setor'] == cod_setor) & (pessoas['tipo_domicilio'] == tipo) & (~pessoas['faixa_etaria'].isin(['0a','0b']))].iloc[:len(id_domicilio_list)].index
        pessoas.loc[index_primeiro_morador, 'id_domicilio'] = np.random.choice(id_domicilio_list, size = len(id_domicilio_list), replace=False)

        #domicilios com numero de moradores definidos
        ids = []
        for row in domicilios_setor.itertuples():
            ids.extend([row.id_domicilio]*(row.num_moradores_int-1))
        index_outros_moradores = pessoas[(pessoas['COD_setor'] == cod_setor) & (pessoas['tipo_domicilio'] == tipo) & (pessoas['id_domicilio'].isna())].iloc[:len(ids)].index
        pessoas.loc[index_outros_moradores, 'id_domicilio'] = np.random.choice(ids, size=len(index_outros_moradores), replace=False)

        #não definidos
        index_outros_moradores = pessoas[(pessoas['COD_setor'] == cod_setor) & (pessoas['tipo_domicilio'] == tipo) & (pessoas['id_domicilio'].isna())].index
        list_dom = domicilios_setor.id_domicilio[domicilios_setor.mais_moradores].to_list()
        if list_dom != []:
            pessoas.loc[index_outros_moradores, 'id_domicilio'] =np.random.choice(list_dom, size=len(index_outros_moradores), replace=True)


C:\Users\carlo\AppData\Local\Temp\ipykernel_30708\3741167280.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'permanente' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  pessoas.loc[index_crianca, 'tipo_domicilio'] = tipo


In [68]:
domicilios['COD_ESPECIE'] = (domicilios.tipo=='coletivo') + 1
domicilios = domicilios.merge(
        pessoas.groupby('id_domicilio').size().reset_index(name='num_moradores_simulacao'), left_on='id_domicilio', right_on='id_domicilio', how='left'   
    ).merge(setores[['CD_SETOR','SITUACAO','NM_UF','NM_MUN','NM_DIST','NM_SUBDIST','NM_BAIRRO','NM_AGLOM','NM_RGINT']],
                  left_on='cod_setor', right_on='CD_SETOR', how='left').drop( columns=['CD_SETOR','num_moradores_int','mais_moradores']       
    ).merge(CNEFE[['COD_ESPECIE','COD_UNICO_ENDERECO','DSC_LOCALIDADE','ponto']], left_on=['COD_ESPECIE','endereco'], right_on=['COD_ESPECIE','COD_UNICO_ENDERECO'], how='left'
    ).drop(columns=['COD_ESPECIE','COD_UNICO_ENDERECO']
    ).merge(simulados_df, left_on=['cod_setor','endereco','tipo','especie'], right_on=['cod_setor','id','tipo','especie'], how='left', suffixes=('', '_simulado')
    ).drop(columns=['id'])
domicilios.loc[domicilios['ponto'].isna(), 'ponto'] = [Point(xy) for xy in domicilios[domicilios['ponto'].isna()]['ponto_simulado'].tolist()]
domicilios.loc[domicilios['fonte']=='simulados', 'endereco'] = [np.nan]*domicilios['fonte'].isin(['simulados']).sum()
domicilios = domicilios.drop(columns=['ponto_simulado'])

In [ ]:
domicilios = domicilios.rename(columns={ 'id_domicilio':'domicilio',
    'SITUACAO':'situacao', 'NM_UF':'uf', 'NM_MUN':'municipio', 'NM_DIST':'distrito', 'NM_SUBDIST':'subdist',
    'NM_BAIRRO':'bairro', 'NM_AGLOM':'aglomerado', 'NM_RGINT':'regiao', 'DSC_LOCALIDADE':'localidade', 
    'num_moradores':'cls_mrdrs','num_moradores_simulacao':'qtd_mrdrs', 'ponto':'localizacao'})
domicilios['cls_mrdrs'] = domicilios['cls_mrdrs'].replace({'sem_num_moradores': 'indef'})

domicilios = domicilios[['domicilio','cod_setor','uf','regiao','municipio','distrito','subdist','bairro','aglomerado',
            'localidade','situacao','fonte', 'endereco','tipo','especie','cls_mrdrs','qtd_mrdrs','localizacao']]

domicilios = gpd.GeoDataFrame(domicilios, geometry='localizacao', crs='EPSG:4326')

C:\Users\carlo\AppData\Local\Temp\ipykernel_30708\1311082557.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  domicilios['cls_mrdrs'].replace({'sem_num_moradores': 'indef'}, inplace=True)


In [70]:
domicilios.loc[domicilios.endereco.notna(),'endereco'] = domicilios.loc[domicilios.endereco.notna(),'endereco'].astype(int).astype(str)

C:\Users\carlo\AppData\Local\Temp\ipykernel_30708\383959112.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['6998898' '6998926' '6998878' ... '182400623' '7005316' '182400719']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  domicilios.loc[domicilios.endereco.notna(),'endereco'] = domicilios.loc[domicilios.endereco.notna(),'endereco'].astype(int).astype(str)


In [71]:
domicilios.to_file('domicilios.shp', 
           driver='ESRI Shapefile',
           encoding='utf-8')

C:\Users\carlo\AppData\Local\Temp\ipykernel_30708\3274116713.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  domicilios.to_file('domicilios.shp',


In [72]:
pessoas.to_csv('individuos.csv', sep=';', index=False)